# Laboratorio 5 - Análisis de paquetes y detección de anomalías
## Security Data Science
### Edwin Andrés Ortega Kou - 22305

### Scapy 

In [1]:
from scapy.all import * 
import pandas as pd 
import numpy as np
import binascii 
import seaborn as sns
sns.set(color_codes=True)
%matplotlib inline

## Análisis de paquetes

### Carga del archivo PCAP

In [2]:
pcap = rdpcap("data/analisis_paquetes.pcap")

print("Cantidad de paquetes:", len(pcap))
print("Tipo de objeto:", type(pcap))

Cantidad de paquetes: 62
Tipo de objeto: <class 'scapy.plist.PacketList'>


### Inspección inicial del archivo

In [3]:
# Ver el primer paquete:
pcap[0]

<Ether  dst=80:37:73:96:9b:db src=88:e9:fe:6a:92:52 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=961 id=1 flags= frag=0 ttl=64 proto=udp chksum=0x52e6 src=10.1.10.53 dst=84.54.22.33 |<UDP  sport=domain dport=domain len=941 chksum=0xf60e |<DNS  id=12 qr=0 opcode=QUERY aa=0 tc=0 rd=1 ra=0 z=0 ad=0 cd=0 rcode=ok qdcount=1 ancount=0 nscount=0 arcount=0 qd=[<DNSQR  qname=b'google.com.' qtype=AAAA unicastresponse=0 qclass=IN |>] |<Raw  load=b'\xef\xbf\xbdPNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01b\x00\x00\x00\xef\xbf\xbd\x08\x06\x00\x00\x00(\xef\xbf\xbdTR\x00\x00:\xef\xbf\xbdIDATx\xef\xbf\xbd\xef\xbf\xbd\t|T\xef\xbf\xbd\xef\xbf\xbd\xef\xbf\xbd\xef\xbf\xbd\xef\xbf\xbd;K\x12\x08;\x08\xef\xbf\xbd\nE\xef\xbf\xbd\xef\xbf\xbd$\x19\xef\xbf\xbd\xef\xbf\xbdZi\xdf\xaa-\xef\xbf\xbd;N2\xef\xbf\xbd\xef\xbf\xbdV\xef\xbf\xbdV\xef\xbf\xbda\xef\xbf\xbdZ\x11!\xef\xbf\xbd\xef\xbf\xbd\x01\xdc\xbbik[\xef\xbf\xbd.ok\xef\xbf\xbde\xef\xbf\xbd\x01\\\xef\xbf\xbd\xef\xbf\xbd]\xef\xbf\xbd-\xef\xbf\xbd\xef\xbf\xbd\xef\x

In [4]:
# Resumen corto del primer paquete
pcap[0].summary()

"Ether / IP / UDP / DNS Qry b'google.com.' / Raw"

### Análisis del primer paquete

In [5]:
ethernet_frame = pcap[0]
ip_packet = ethernet_frame.payload
segment = ip_packet.payload
data = segment.payload

print("Capa de enlace:", type(ethernet_frame))
print("Capa de red:", type(ip_packet))
print("Capa de transporte:", type(segment))
print("Payload:", type(data))

Capa de enlace: <class 'scapy.layers.l2.Ether'>
Capa de red: <class 'scapy.layers.inet.IP'>
Capa de transporte: <class 'scapy.layers.inet.UDP'>
Payload: <class 'scapy.layers.dns.DNS'>


### Visualización en hexadecimal

In [6]:
hexdump(pcap[0])

0000  80 37 73 96 9B DB 88 E9 FE 6A 92 52 08 00 45 00  .7s......j.R..E.
0010  03 C1 00 01 00 00 40 11 52 E6 0A 01 0A 35 54 36  ......@.R....5T6
0020  16 21 00 35 00 35 03 AD F6 0E 00 0C 01 00 00 01  .!.5.5..........
0030  00 00 00 00 00 00 06 67 6F 6F 67 6C 65 03 63 6F  .......google.co
0040  6D 00 00 1C 00 01 EF BF BD 50 4E 47 0D 0A 1A 0A  m........PNG....
0050  00 00 00 0D 49 48 44 52 00 00 01 62 00 00 00 EF  ....IHDR...b....
0060  BF BD 08 06 00 00 00 28 EF BF BD 54 52 00 00 3A  .......(...TR..:
0070  EF BF BD 49 44 41 54 78 EF BF BD EF BF BD 09 7C  ...IDATx.......|
0080  54 EF BF BD EF BF BD EF BF BD EF BF BD EF BF BD  T...............
0090  3B 4B 12 08 3B 08 EF BF BD 0A 45 EF BF BD EF BF  ;K..;.....E.....
00a0  BD 24 19 EF BF BD EF BF BD 5A 69 DF AA 2D EF BF  .$.......Zi..-..
00b0  BD 3B 4E 32 EF BF BD EF BF BD 56 EF BF BD 56 EF  .;N2......V...V.
00c0  BF BD 61 EF BF BD 5A 11 21 EF BF BD EF BF BD 01  ..a...Z.!.......
00d0  DC BB 69 6B 5B EF BF BD 2E 6F 6B EF BF BD 65 EF  ..ik[....

### Campos disponibles en el paquete

In [7]:
ls(pcap[0])

dst        : DestMACField                        = '80:37:73:96:9b:db' ('None')
src        : SourceMACField                      = '88:e9:fe:6a:92:52' ('None')
type       : XShortEnumField                     = 2048            ('36864')
--
version    : BitField  (4 bits)                  = 4               ('4')
ihl        : BitField  (4 bits)                  = 5               ('None')
tos        : XByteField                          = 0               ('0')
len        : ShortField                          = 961             ('None')
id         : ShortField                          = 1               ('1')
flags      : FlagsField                          = <Flag 0 ()>     ('<Flag 0 ()>')
frag       : BitField  (13 bits)                 = 0               ('0')
ttl        : ByteField                           = 64              ('64')
proto      : ByteEnumField                       = 17              ('0')
chksum     : XShortField                         = 21222           ('None')
src       

### Resumen del paquete analizado

In [8]:
print(pcap[0].summary())

Ether / IP / UDP / DNS Qry b'google.com.' / Raw


## Otras funciones

In [ ]:
# high-level functions are already coded
lsc()

## Convertir PCAP a un DataFrame

Scapy es una librería muy versatil, pero sus estructuras de datos no son tan fáciles de manipular como un data frame

In [ ]:
# Collect field names from IP/TCP/UDP (These will be columns in DF)
ip_fields = [field.name for field in IP().fields_desc]
tcp_fields = [field.name for field in TCP().fields_desc]
udp_fields = [field.name for field in UDP().fields_desc]

dataframe_fields = ip_fields + ['time'] + tcp_fields + ['payload','payload_raw','payload_hex']

# Create blank DataFrame
df = pd.DataFrame(columns=dataframe_fields)
for packet in pcap[IP]:
    # Field array for each row of DataFrame
    field_values = []
    # Add all IP fields to dataframe
    for field in ip_fields:
        if field == 'options':
            # Retrieving number of options defined in IP Header
            field_values.append(len(packet[IP].fields[field]))
        else:
            field_values.append(packet[IP].fields[field])
    
    field_values.append(packet.time)
    
    layer_type = type(packet[IP].payload)
    for field in tcp_fields:
        try:
            if field == 'options':
                field_values.append(len(packet[layer_type].fields[field]))
            else:
                field_values.append(packet[layer_type].fields[field])
        except:
            field_values.append(None)
    
    # Append payload
    field_values.append(len(packet[layer_type].payload))
    field_values.append(packet[layer_type].payload.original)
    field_values.append(binascii.hexlify(packet[layer_type].payload.original))
    # Add row to DF
    df_append = pd.DataFrame([field_values], columns=dataframe_fields)
    df = pd.concat([df, df_append], axis=0)

# Reset Index
df = df.reset_index()
# Drop old index column
df = df.drop(columns="index")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df

In [ ]:
pcap[IP]

In [ ]:
print(len(pcap))


## Estadísticas

In [ ]:
# Top Source Adddress
# Para datos categóricos, describe devuelve conteo, valores distintos, valor más frecuente (top) y su frecuencia
print("# Top Source Address")
print(df['src'].describe(),'\n\n') 

frequent_address = df['src'].describe()['top']

# Who is the top address speaking to
print("# Who is Top Address Speaking to?")
print(df[df['src'] == frequent_address]['dst'].unique(),"\n\n")



In [ ]:
# Unique Source Addresses
print("Unique Source Addresses")
print(df['src'].unique())

print()

# Unique Destination Addresses
print("Unique Destination Addresses")
print(df['dst'].unique())

In [ ]:
# Group by Source Address and Payload Sum
source_addresses = df.groupby("src")['payload'].sum()
source_addresses.plot(kind='barh',title="Addresses Sending Payloads",figsize=(8,5))

In [ ]:
features = df.columns.tolist()
features

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats


important_features = ['len', 'payload']


# ---- Panel 1: Histograms with Gaussian fit overlay ----
fig, axes = plt.subplots(1, 2, figsize=(24, 4))

for ax, feat in zip(axes, important_features):
    data = df[feat].values
    ax.hist(data, bins=35, density=True, alpha=0.6, color='#4A90D9', edgecolor='white')

    # Overlay the best-fit Gaussian curve
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'N({mu:.0f}, {sigma:.0f}²)')

    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Histogram vs. Gaussian Fit',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()